# A/B Test Analysis: Does a New Landing Page Increase Conversions?

**Question:** Does the new landing page (B) give a higher conversion rate than the old page (A)?

**Data:** [A/B testing dataset on Kaggle](https://www.kaggle.com/datasets/zhangluyuan/ab-testing), included in this repository as `ab_data.csv`.

**Plan:**
1. Load and clean the data
2. Baseline: compare the raw conversion rates
3. Two-proportion z-test and 95% confidence interval
4. Sample-size calculation
5. Checks on the method: A/A test, peeking and stability
6. Sanity checks
7. Conclusion

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Settings
DATA_PATH = "ab_data.csv"
ALPHA = 0.05      # significance level: we accept a 5% chance of a false positive
POWER = 0.80      # 80% chance of detecting a real effect of MIN_LIFT
MIN_LIFT = 0.005  # smallest useful lift: 0.5 percentage points (e.g. 12.0% -> 12.5%)
SEED = 42         # fixed random seed so every run gives the same numbers

## 1. Load and clean the data

Two kinds of rows would make the comparison unfair:
- **Mismatched rows:** a `control` user who saw the new page, or a `treatment` user who saw the old page. We can't tell which version they really had.
- **Duplicate users:** each user must be counted once, otherwise the observations are not independent. We keep the first visit.

In [ ]:
def clean_data(df):
    """Remove mismatched rows and duplicate users. Returns (clean data, report)."""
    report = {"rows_raw": len(df)}

    matched = (
        ((df["group"] == "control") & (df["landing_page"] == "old_page"))
        | ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
    )
    report["rows_mismatched"] = int((~matched).sum())
    df = df[matched]

    duplicated = df["user_id"].duplicated(keep="first")
    report["duplicate_users_removed"] = int(duplicated.sum())
    df = df[~duplicated].reset_index(drop=True)

    report["rows_clean"] = len(df)
    return df, report


raw = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df, report = clean_data(raw)
report

## 2. Baseline: compare the raw conversion rates

The simplest possible answer: which group converted more often? The error bars show the 95% range for each rate.

In [ ]:
summary = df.groupby("group")["converted"].agg(users="count", conversions="sum", rate="mean")
summary

In [ ]:
rates = summary["rate"]
errors = 1.96 * np.sqrt(rates * (1 - rates) / summary["users"])  # 95% CI half-width

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Control (old page)", "Treatment (new page)"], rates,
       yerr=errors, capsize=8, color=["#7f8c8d", "#2e86c1"])
ax.set_ylabel("Conversion rate")
ax.set_ylim(0.10, 0.13)  # zoomed in so the small difference is visible
ax.set_title("Conversion rate by group (95% CI)")
plt.show()

## 3. Two-proportion z-test and confidence interval

- **H₀:** rate_B = rate_A
- **H₁:** rate_B ≠ rate_A (two-sided)

The **z-test** uses the *pooled* rate for the standard error, because under H₀ both groups share one true rate.
The **confidence interval** uses the *unpooled* standard error, because there we do not assume the rates are equal.

In [ ]:
def two_proportion_ztest(conv_a, n_a, conv_b, n_b):
    """Two-sided z-test for the difference between two proportions. Returns (z, p_value)."""
    p_a, p_b = conv_a / n_a, conv_b / n_b
    p_pool = (conv_a + conv_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se
    p_value = 2 * norm.sf(abs(z))  # sf = 1 - cdf, the area in one tail
    return z, p_value


def diff_confidence_interval(conv_a, n_a, conv_b, n_b, alpha=0.05):
    """Confidence interval for rate_B - rate_A. Returns (diff, low, high)."""
    p_a, p_b = conv_a / n_a, conv_b / n_b
    diff = p_b - p_a
    se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z_crit = norm.ppf(1 - alpha / 2)  # 1.96 for a 95% interval
    return diff, diff - z_crit * se, diff + z_crit * se


conv_a, n_a = summary.loc["control", "conversions"], summary.loc["control", "users"]
conv_b, n_b = summary.loc["treatment", "conversions"], summary.loc["treatment", "users"]

z, p_value = two_proportion_ztest(conv_a, n_a, conv_b, n_b)
diff, ci_low, ci_high = diff_confidence_interval(conv_a, n_a, conv_b, n_b, ALPHA)

print(f"Difference (B - A): {diff * 100:+.3f} percentage points")
print(f"95% CI:             [{ci_low * 100:+.3f}, {ci_high * 100:+.3f}] percentage points")
print(f"z = {z:.3f}, p = {p_value:.4f}, significant at {ALPHA}: {p_value < ALPHA}")
print(f"Is a {MIN_LIFT * 100:.1f} pp lift still inside the CI? {ci_high >= MIN_LIFT}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.errorbar(diff, 0, xerr=[[diff - ci_low], [ci_high - diff]],
            fmt="o", capsize=8, color="#2e86c1")
ax.axvline(0, color="black", linestyle="--", label="No difference")
ax.axvline(MIN_LIFT, color="#c0392b", linestyle=":", label="Minimum useful lift")
ax.set_yticks([])
ax.set_xlabel("Treatment rate minus control rate")
ax.set_title("95% confidence interval for the difference")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## 4. Sample size: was the test big enough?

Users needed **per group** to detect a lift of `MIN_LIFT` with a two-sided test:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \,\big(p_1(1-p_1) + p_2(1-p_2)\big)}{(p_2 - p_1)^2}$$

In [ ]:
def required_sample_size(baseline_rate, min_lift, alpha=0.05, power=0.80):
    """Users needed per group to detect a lift of min_lift."""
    p1, p2 = baseline_rate, baseline_rate + min_lift
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    n = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2
    return int(np.ceil(n))


baseline_rate = conv_a / n_a
n_needed = required_sample_size(baseline_rate, MIN_LIFT, ALPHA, POWER)
n_have = int(summary["users"].min())
print(f"Users needed per group for a {MIN_LIFT * 100:.1f} pp lift: {n_needed:,}")
print(f"Users we have in the smaller group:        {n_have:,}")

In [ ]:
lifts = np.linspace(0.002, 0.02, 50)
needed = [required_sample_size(baseline_rate, l, ALPHA, POWER) for l in lifts]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lifts * 100, needed, color="#2e86c1")
ax.axhline(n_have, color="#27ae60", linestyle="--", label="Users we have per group")
ax.set_yscale("log")
ax.set_xlabel("Lift to detect (percentage points)")
ax.set_ylabel("Users needed per group")
ax.set_title("Smaller lifts need far more users")
ax.legend()
plt.show()

## 5. Checks on the method

**Note on evaluation:** This project uses a statistical test, not a prediction model, so there is no held-out test set, cross-validation or set of wrong predictions. Instead, the method is evaluated by:
1. **A/A tests**, to check the false-positive rate is about 5% (5a)
2. **A peeking simulation**, to show how stopping early causes errors (5b)
3. **A stability check**, repeating the test on the first and second half of the test period (5c)
4. **A sample-size check**, to confirm the test had enough data (section 4)
5. **Sanity checks** of the formulas on known cases (section 6)

### 5a. A/A test: how often does the test find a difference that isn't there?

We take only control users (who all saw the **same** page) and split them randomly into two fake groups, 1,000 times.
There is no real difference, so every "significant" result is a false positive. About 5% should be below 0.05.

In [ ]:
def aa_test_pvalues(converted, n_sims=1000, seed=42):
    """p-values from n_sims random A/A splits of the same users."""
    rng = np.random.default_rng(seed)
    converted = np.asarray(converted)
    p_values = np.empty(n_sims)
    for i in range(n_sims):
        in_b = rng.integers(0, 2, size=len(converted)).astype(bool)  # coin flip per user
        a, b = converted[~in_b], converted[in_b]
        _, p_values[i] = two_proportion_ztest(a.sum(), len(a), b.sum(), len(b))
    return p_values


control = df[df["group"] == "control"]
aa_p = aa_test_pvalues(control["converted"], n_sims=1000, seed=SEED)
aa_fp_rate = (aa_p < ALPHA).mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(aa_p, bins=20, range=(0, 1), color="#7f8c8d", edgecolor="white")
ax.axvline(ALPHA, color="#c0392b", linestyle="--", label=f"alpha = {ALPHA}")
ax.set_xlabel("p-value")
ax.set_ylabel("Number of A/A tests")
ax.set_title(f"A/A tests: {aa_fp_rate:.1%} false positives")
ax.legend()
plt.show()

### 5b. Peeking: what if we had checked the result every day?

First, the p-value of the **real** test if it had been stopped at the end of each day.

In [ ]:
def pvalues_over_time(df):
    """p-value of the test using all data up to the end of each day."""
    daily = (
        df.assign(day=df["timestamp"].dt.date)
        .groupby(["day", "group"])["converted"]
        .agg(["count", "sum"])
        .unstack("group", fill_value=0)
        .cumsum()  # running totals: everything seen up to that day
    )
    rows = []
    for day, r in daily.iterrows():
        _, p = two_proportion_ztest(
            r[("sum", "control")], r[("count", "control")],
            r[("sum", "treatment")], r[("count", "treatment")],
        )
        rows.append({"day": day, "users": int(r["count"].sum()), "p_value": p})
    return pd.DataFrame(rows)


over_time = pvalues_over_time(df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(over_time["day"], over_time["p_value"], marker="o", color="#2e86c1")
ax.axhline(ALPHA, color="#c0392b", linestyle="--", label=f"alpha = {ALPHA}")
ax.set_ylabel("p-value")
ax.set_title("p-value if the test had stopped on each day")
ax.legend()
fig.autofmt_xdate()
plt.show()

Now the danger: 500 A/A tests where we check **every day** and stop at the first p < 0.05.
Each extra look is another chance for a false positive.

In [ ]:
def peeking_false_positive_rate(df_one_group, n_sims=500, alpha=0.05, seed=42):
    """False-positive rate checking only at the end vs checking every day."""
    rng = np.random.default_rng(seed)
    days = df_one_group["timestamp"].dt.date.to_numpy()
    converted = df_one_group["converted"].to_numpy()
    unique_days = np.unique(days)                   # sorted list of days
    day_index = np.searchsorted(unique_days, days)  # day number of each user

    end_only, any_day = 0, 0
    for _ in range(n_sims):
        fake_group = rng.integers(0, 2, size=len(converted))
        users = np.zeros((len(unique_days), 2))
        convs = np.zeros((len(unique_days), 2))
        np.add.at(users, (day_index, fake_group), 1)          # users per day per group
        np.add.at(convs, (day_index, fake_group), converted)  # conversions per day per group
        users, convs = users.cumsum(axis=0), convs.cumsum(axis=0)

        _, p = two_proportion_ztest(convs[:, 0], users[:, 0], convs[:, 1], users[:, 1])
        end_only += p[-1] < alpha     # only the final day counts
        any_day += (p < alpha).any()  # stop as soon as any day looks significant
    return end_only / n_sims, any_day / n_sims


fp_end, fp_peek = peeking_false_positive_rate(control, n_sims=500, alpha=ALPHA, seed=SEED)
print(f"False positives, checking once at the end: {fp_end:.1%}")
print(f"False positives, checking every day:       {fp_peek:.1%}")

### 5c. Stability check: first half vs second half

This is the closest thing to a held-out test. We split the test period at its midpoint and run the same z-test on each half separately.
If both halves point the same way, the result is stable over time and not caused by a few unusual days.

In [ ]:
midpoint = df["timestamp"].min() + (df["timestamp"].max() - df["timestamp"].min()) / 2

rows = []
for name, part in [("First half", df[df["timestamp"] < midpoint]),
                   ("Second half", df[df["timestamp"] >= midpoint])]:
    s = part.groupby("group")["converted"].agg(users="count", conversions="sum")
    ca, na = s.loc["control", "conversions"], s.loc["control", "users"]
    cb, nb = s.loc["treatment", "conversions"], s.loc["treatment", "users"]
    _, p = two_proportion_ztest(ca, na, cb, nb)
    d, lo, hi = diff_confidence_interval(ca, na, cb, nb, ALPHA)
    rows.append({"period": name, "users": int(na + nb),
                 "rate_control_%": round(ca / na * 100, 2),
                 "rate_treatment_%": round(cb / nb * 100, 2),
                 "diff_pp": round(d * 100, 3),
                 "ci_low_pp": round(lo * 100, 3), "ci_high_pp": round(hi * 100, 3),
                 "p_value": round(p, 4)})

print(f"Split at: {midpoint}")
pd.DataFrame(rows)

## 6. Sanity checks on the functions

Small known cases, to make sure the formulas behave as expected. If any check fails, the cell raises an error.

In [ ]:
# Identical groups: no difference, p-value of 1
z_test, p_test = two_proportion_ztest(100, 1000, 100, 1000)
assert abs(z_test) < 1e-9 and abs(p_test - 1) < 1e-9

# A big difference (10% vs 20%) must be significant
assert two_proportion_ztest(100, 1000, 200, 1000)[1] < 0.001

# The CI must contain the observed difference
d, lo, hi = diff_confidence_interval(100, 1000, 150, 1000)
assert lo < d < hi

# A smaller lift needs more users; 10% -> 12% needs about 3,840 per group
assert required_sample_size(0.12, 0.005) > required_sample_size(0.12, 0.01)
assert abs(required_sample_size(0.10, 0.02) - 3839) <= 5

# Cleaning: every user appears once and every row is matched
assert df["user_id"].is_unique
assert ((df["group"] == "control") == (df["landing_page"] == "old_page")).all()

print("All checks passed")

## 7. Conclusion

- **The new page did not win.** It converted 11.88% of users against 12.04% for the old page, a difference of −0.16 percentage points (z = −1.31, p = 0.19). This is not statistically significant at α = 0.05.
- **Any lift is too small to matter.** The 95% CI for the difference is [−0.39, +0.08] percentage points. It does not reach the 0.5 pp minimum useful lift, so if the new page helps at all, the gain is too small to justify the change.
- **The test was big enough.** Detecting a 0.5 pp lift with 80% power needs about 67,700 users per group; the test had about 145,000. The null result is not caused by too little data.
- **The method behaves correctly.** In 1,000 A/A tests, 5.3% were "significant", close to the expected 5%.
- **Peeking is dangerous.** Checking once at the end gave 5.4% false positives, but checking every day and stopping at the first p < 0.05 gave 28.4%, about five times more. The test length must be fixed in advance.
- **The result is stable over time.** Splitting the test period at its midpoint gave the same answer in both halves: the new page was slightly worse (−0.04 pp, p = 0.83, and −0.28 pp, p = 0.10). Neither half was significant, and neither CI reached the 0.5 pp minimum useful lift. The two halves differ a little, but their CIs overlap heavily, so the gap is normal random variation.

**Recommendation:** keep the old page.

## Limitations
- The test ran for about 3 weeks, so long-term effects (such as novelty wearing off) are not measured.
- The dataset has no business context, so the 0.5 pp "useful lift" is an assumption.
- Only one metric (conversion) is analysed.
- Users who visited more than once are counted on their first visit only.